# Loan Default Prediction — leakage-safe evaluation

Custom CatBoost experiment on the public loan-default dataset. Hyperparameters use training folds only; the decision threshold uses validation only. A final test set is used after both decisions are frozen. Run from this folder after installing requirements.

The original notebook’s threshold comparisons used test labels; its saved metrics are superseded. F1 is a development objective because no lending error costs were supplied.

In [1]:
import os, json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from loan_pipeline import run

CSV = Path(os.environ.get("LOAN_CSV", "Loan_default.csv"))
OUTPUT = Path(os.environ.get("LOAN_OUTPUT", "artifacts"))
CANDIDATES = int(os.environ.get("LOAN_CANDIDATES", "4"))
FOLDS = int(os.environ.get("LOAN_FOLDS", "3"))
THREADS = int(os.environ.get("LOAN_THREADS", "2"))
assert CSV.is_file(), f"Download the dataset first: {CSV}"

## Data and experiment design

Rows are split 60/20/20 with stratification and seed 42. IDs are excluded from features. CatBoost handles numeric missing values; categorical missing values get an explicit token. Income ratios are computed without fitted statistics. ROC-AUC selects hyperparameters in stratified training CV. Validation F1 selects a threshold on a fixed 0.01–0.99 grid; ties prefer the higher threshold. The fitted model is not refit on validation.

This random split assumes independent borrowers. If repeated borrowers or application timestamps become available, replace it with group/time splits before claiming realistic deployment performance.

In [2]:
preview = pd.read_csv(CSV, nrows=5)
print("Columns:", preview.columns.tolist())
preview

Columns: ['LoanID', 'Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed', 'NumCreditLines', 'InterestRate', 'LoanTerm', 'DTIRatio', 'Education', 'EmploymentType', 'MaritalStatus', 'HasMortgage', 'HasDependents', 'LoanPurpose', 'HasCoSigner', 'Default']


       LoanID  Age  Income  ... LoanPurpose HasCoSigner  Default
0  I38PQUQS96   56   85994  ...       Other         Yes        0
1  HPSK72WA7R   69   50432  ...       Other         Yes        0
2  C1OZ6DPJ8Y   46   84208  ...        Auto          No        1
3  V2KKSFM3UN   32   31713  ...    Business          No        0
4  EY08JDHTZP   60   20437  ...        Auto          No        0

[5 rows x 18 columns]

In [3]:
report = run(CSV, OUTPUT, n_iter=CANDIDATES, folds=FOLDS, threads=THREADS)
print("Partitions:", report["split"])
print("Chosen parameters:", report["selection"]["best_parameters"])
print("Validation-selected threshold:", report["selection"]["validation"]["threshold"])

Partitions: {'train': 153207, 'validation': 51070, 'test': 51070}
Chosen parameters: {'learning_rate': 0.08, 'l2_leaf_reg': 3, 'iterations': 200, 'depth': 4, 'auto_class_weights': 'Balanced'}
Validation-selected threshold: 0.61


## Final holdout results

The next cells only describe the frozen model. Do not adjust features, hyperparameters or threshold in response to these test results. A new holdout is needed for further selection.

In [4]:
columns = ["roc_auc", "average_precision", "accuracy", "precision", "recall", "f1", "threshold"]
pd.DataFrame({name: {key: report[name][key] for key in columns}
              for name in ["test", "test_at_0_5", "majority_baseline_test"]}).T

                        roc_auc  average_precision  ...        f1  threshold
test                    0.75883           0.331802  ...  0.371113       0.61
test_at_0_5             0.75883           0.331802  ...  0.340752       0.50
majority_baseline_test  0.50000           0.116135  ...  0.000000       0.50

[3 rows x 7 columns]

In [5]:
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(report["test"]["confusion_matrix"], annot=True, fmt="d", cmap="Blues",
            xticklabels=["No default", "Default"], yticklabels=["No default", "Default"], ax=ax)
ax.set(xlabel="Predicted", ylabel="Actual", title="Final test confusion matrix")
fig.tight_layout()
fig.savefig(OUTPUT / "confusion_matrix.png", dpi=150)
plt.show()

<Figure size 500x400 with 2 Axes>

In [6]:
pd.DataFrame(report["test"]["classification_report"]).T

              precision    recall  f1-score       support
0              0.929881  0.829083  0.876594  45139.000000
1              0.287232  0.524195  0.371113   5931.000000
accuracy       0.793675  0.793675  0.793675      0.793675
macro avg      0.608557  0.676639  0.623854  51070.000000
weighted avg   0.855247  0.793675  0.817890  51070.000000

## Export and limitations

`artifacts/` contains the CatBoost model, feature contract, selected threshold, split indices, validation curve, CV results and evaluation JSON. Inference must call `prepare_features` and compare positive-class probabilities against the saved threshold. Accuracy alone is misleading for this imbalanced target; read PR/ROC metrics and the precision/recall trade-off together. The result is an educational benchmark, not an approved lending policy. No fairness, drift or calibrated-risk claim is made.

In [7]:
from catboost import CatBoostClassifier
from loan_pipeline import prepare_features
metadata = json.loads((OUTPUT / "model_metadata.json").read_text())
loaded = CatBoostClassifier()
loaded.load_model(str(OUTPUT / "loan_default_model.cbm"))
probability = loaded.predict_proba(prepare_features(preview))[:, 1]
pd.DataFrame({"default_probability": probability,
              "predicted_default": (probability >= metadata["threshold"]).astype(int)})

   default_probability  predicted_default
0             0.194864                  0
1             0.178329                  0
2             0.620699                  1
3             0.533913                  0
4             0.274133                  0